# Notebook 06: feature-space projections on the demonstrator capture

The capture pipeline emits 21 spectral and statistical features
per trace. This is a superset of the band-localised feature set
used by the paper's EM hypothesis rule (4 features over 3 bands,
Section 2.2 of the paper). This notebook visualises how the
wider raw feature set separates `exit_code == 0` from
`exit_code != 0` traces in two-dimensional projections (PCA and
t-SNE) and embeds the corresponding reference figures emitted by
the production capture pipeline.

Two scope reminders from notebook `05` apply here:

1. The split visualised here is by exit code, not by the
   paper's EM-anomalous criterion.
2. The picoc capture used here is from a different campaign
   than the picoc campaign in the paper, so cluster counts and
   per-class proportions do not match the paper.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

FEATURES_CSV = Path("..") / "data" / "picoc_features.csv"
FIGURES_DIR = Path("..") / "figures" / "em_evidence"

df = pd.read_csv(FEATURES_CSV)
feature_cols = ['rms', 'mean', 'std_dev', 'crest_factor', 'peak_to_peak',
                'entropy', 'peak_count', 'kurtosis', 'skewness', 'zcr',
                'energy_low', 'energy_mid', 'energy_high',
                'peak_freq', 'peak_magnitude', 'spectral_entropy',
                'spectral_crest', 'harmonic_distortion',
                'spectral_flatness', 'spectral_rolloff', 'hnr']
X = df[feature_cols].fillna(0).values
y = df["is_anomalous"].values
print(f"Feature matrix shape: {X.shape}")
print(f"Anomalous traces: {int(y.sum())}")


## Standardise features


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Mean of scaled features (should be ~0):", np.round(X_scaled.mean(axis=0), 6)[:5], "...")
print("Std of scaled features  (should be ~1):", np.round(X_scaled.std(axis=0), 6)[:5], "...")


## PCA in 2D


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
evr = pca.explained_variance_ratio_
print(f"Explained variance ratio (PC1, PC2): {evr[0]:.3f}, {evr[1]:.3f}")
print(f"Cumulative explained variance:       {evr.sum():.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
for label, colour, name in [(False, "#1f4e79", "normal"), (True, "#a23b3b", "anomalous")]:
    mask = (y == label)
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.5, color=colour, label=name)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}% var)")
ax.set_title("PCA projection of EM features")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## Scree plot of the full PCA decomposition

PC1 and PC2 jointly account for ~47% of the variance; the
remainder is distributed across the trailing principal
components. The scree plot below shows that the spectrum
decays smoothly without a sharp elbow, which is consistent with
a feature set in which several spectral descriptors carry
complementary information rather than one dominant axis.


In [ ]:
from sklearn.decomposition import PCA as _PCA_full
_pca_full = _PCA_full(random_state=42)
_pca_full.fit(X_scaled)
evr_full = _pca_full.explained_variance_ratio_
evr_cum = evr_full.cumsum()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(evr_full) + 1), evr_full, color="#1f4e79", label="per-PC variance")
ax2 = ax.twinx()
ax2.plot(range(1, len(evr_full) + 1), evr_cum, color="#a23b3b", marker="o", label="cumulative")
ax.set_xlabel("Principal component index")
ax.set_ylabel("Explained variance ratio (per PC)")
ax2.set_ylabel("Cumulative explained variance")
ax.set_title("Scree plot of PCA on the 21 raw features")
ax.grid(True, axis="y", linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()

for i, (v, c) in enumerate(zip(evr_full, evr_cum), start=1):
    print(f"PC{i:>2}: {v*100:5.2f}%   cumulative: {c*100:5.2f}%")


## t-SNE in 2D

t-SNE is initialised from the PCA solution to keep the
projection reproducible across runs.


In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 5))
for label, colour, name in [(False, "#1f4e79", "normal"), (True, "#a23b3b", "anomalous")]:
    mask = (y == label)
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=8, alpha=0.5, color=colour, label=name)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.set_title("t-SNE projection of EM features (perplexity=30)")
ax.legend()
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## Reference figure: PCA analysis from the EM pipeline

Reference PCA produced by the EM analysis pipeline against the
same capture. Compare with the live reproduction above; minor
visual differences are expected due to algorithmic variations
and parameter choices in the production pipeline.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_pca_analysis.pdf"), width=800, height=600))


## Reference figure: cluster comparison

Side-by-side comparison of cluster assignments produced under
different clustering settings. Used to argue that the
anomalous regime is structurally separable in feature space.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_cluster_comparison.pdf"), width=800, height=600))


## Reference figure: 3D anomaly view

Three-dimensional projection of anomalous traces against the
normal-trace background, used to visualise sub-cluster
structure inside the anomalous regime.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_anomaly_3d.pdf"), width=800, height=600))


## Reference figure: classification with zoom

Classifier decision surface in the projected feature space,
with a zoom panel onto the boundary region between normal and
anomalous traces.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_classification_zoom.pdf"), width=800, height=600))


## Reference figure: anomaly clusters

Per-cluster aggregation of the anomalous traces, exposing the
internal subdivision of the anomalous regime that motivates the
subsystem-level interpretation in the agreement step.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_anomaly_clusters.pdf"), width=800, height=600))
